# Load the pdf files and split them into chunks

In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader

directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)

docs = directory_loader.load()

print(docs) 

[Document(metadata={'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2025-10-17T19:51:23+00:00', 'source': 'data/rag-test.pdf', 'file_path': 'data/rag-test.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Amit Shrigondekar', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T19:51:23+00:00', 'trapped': '', 'modDate': "D:20251017195123+00'00'", 'creationDate': "D:20251017195123+00'00'", 'page': 0}, page_content='This is a text to prove that RAG works.  \nAmit Shrigondekar is the full stack engineer. His skills are backend specifically in node, \njava, bash , Kubernetes etc. \nAmit likes to spend time building things that automates boring repetitive error-prone \ntedious work so that it can be done efficiently. \nAmit does not like to be told details on how he need to perform his role at work. Amit likes \nto explore, design, research himself to figure out the right way to architect and build things.'), Document(metadata={'producer': 'macOS Version 15.

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
split_docs = text_splitter.split_documents(docs)

print(split_docs)

len(split_docs)

[Document(metadata={'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2025-10-17T19:51:23+00:00', 'source': 'data/rag-test.pdf', 'file_path': 'data/rag-test.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Amit Shrigondekar', 'subject': '', 'keywords': '', 'moddate': '2025-10-17T19:51:23+00:00', 'trapped': '', 'modDate': "D:20251017195123+00'00'", 'creationDate': "D:20251017195123+00'00'", 'page': 0}, page_content='This is a text to prove that RAG works.  \nAmit Shrigondekar is the full stack engineer. His skills are backend specifically in node, \njava, bash , Kubernetes etc. \nAmit likes to spend time building things that automates boring repetitive error-prone \ntedious work so that it can be done efficiently. \nAmit does not like to be told details on how he need to perform his role at work. Amit likes \nto explore, design, research himself to figure out the right way to architect and build things.'), Document(metadata={'producer': 'macOS Version 15.

201

# Create embeddings

In [3]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
# create Qdrant Vector Store
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance,VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="rag_collection_name",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="rag_collection_name",
    embedding=embeddings,
)

In [5]:
# Add documents to the vector store
vector_store.add_documents(documents=split_docs)

['319e4a87c21c42c5b8fb58f556269a95',
 'e53980120caa48b98351ff6183c8c0c6',
 '074babffe1c3461491b6af9682be20b6',
 '23708b8d2bcb45eb923519cc44afda33',
 'dc1b013d9cc345a091abede13952e497',
 '6cb1fab1977948c79c48c26285a00741',
 'e15ecd04fb09453ca7a334d024e8af70',
 '3fb2ae500eca4171a51de88828537e1c',
 '231e1b688ba34968ac172e0b0a58eb62',
 '4d17fb9b70a74c0f9b1c0ae14ccc510b',
 '8cb98f3a547c48c497bbbc294feacc29',
 '91c02d62597049389dd74efdf4fa0f73',
 '4f86ac47bf1043259bfb02d4359820d9',
 '76ec0c446f474929a5f7c6ad0b00be12',
 'c70c76ab5a9b4d47b6dae1c2537246a4',
 'fba9586a105f450fbf4ee097161119f7',
 '3ce3142e255a492ea27cfc50d5789a74',
 'c9ec6c0e1dba4987bb43b3eba0c63cfc',
 'dc6ebc4382bf4c6a93e69bda6e548cca',
 '67b59701af3f498997c7cb05a2a9269d',
 '9bdb1f3c851b402c800e2881e8b0b412',
 '843905aa2083414a990c2bf170f0c99b',
 '3c7eedae1c0c4a0c8e6d6577775d9197',
 'be9be9c86ab64e14b5146a046afa4b9c',
 'd580d7a31af04e7fbd71d989a130d8b9',
 '7eefb6b3646a402283175039e9ca31f5',
 '78740f1a2b2f4af481b6a4e48fe3d420',
 

In [6]:
# Lets create a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})



In [7]:
retriever.invoke("what is the main purpose of the document?")

[Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:32+00:00', 'source': 'data/howpeopleuseai.pdf', 'file_path': 'data/howpeopleuseai.pdf', 'total_pages': 64, 'format': 'PDF 1.6', 'title': 'How People Use ChatGPT', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-15T10:32:36-04:00', 'trapped': '', 'modDate': "D:20250915103236-04'00'", 'creationDate': 'D:20250912200532Z', 'page': 21, '_id': '7fcc74003a1b48dd9768ed1442fbe96b', '_collection_name': 'rag_collection_name'}, page_content='at work appears to be focused on two broad functions: 1) obtaining, documenting, and interpreting\ninformation; and 2) making decisions, giving advice, solving problems, and thinking creatively.\n20'),
 Document(metadata={'producer': 'macOS Version 15.4.1 (Build 24E263) Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-09-12T20:05:3

In [8]:
# produce node for the retrieval
def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": retrieved_docs}

In [9]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
    You are a helpful assistant who can answer questions based on the following context only:
    If you cannot answer the question based on the context - you must say "I don't know".
    
    ### Context: 
    {context}
    
    ### Question: 
    {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

In [10]:
# Create LLM instance
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


llm = ChatOpenAI(model="gpt-4.1-mini")



In [11]:
# Create a generator

def generate(state):
    docs_content="\n\n".join( doc.page_content for doc in state["context"])
    message = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(message)
    return {"response": response.content}

In [12]:
# build a state 
from typing_extensions import TypedDict, List
from langchain_core.documents import Document

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

In [13]:
# Lets build a graph 
from langgraph.graph import START,StateGraph

# 
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph=graph_builder.compile()

In [14]:
# Run the graph
response = graph.invoke({"question": "who is Amit Shrigondekar?"})

print(response["response"])


Amit Shrigondekar is a full stack engineer with skills in backend development, specifically in node, java, bash, Kubernetes, etc. He likes to spend time building things that automate boring, repetitive, error-prone, and tedious work to make it more efficient. Amit does not like to be told details on how to perform his role at work; instead, he prefers to explore, design, and research himself to figure out the right way to architect and build things.


In [15]:
# Lets create some tools 

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

arxiv_tool = ArxivQueryRun()
tavily_tool = TavilySearchResults(max_results=5)




/var/folders/0j/52nrhqqd6hq140jsz5rrd5km0000gq/T/ipykernel_49193/2375597365.py:7: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


## ⚠️ Critical Fix: ai_rag_tool Now Returns Contexts

**Problem**: The original `ai_rag_tool` only returned the final answer, throwing away the retrieved document chunks.

**Solution**: Modified to return JSON with BOTH:
- `answer`: The generated response
- `contexts`: List of retrieved document chunks

This allows RAGAS to evaluate whether the answer is grounded in the retrieved contexts.


In [ ]:
# lets create a tool for the graph  
from langchain_core.tools import tool
import json

@tool
def ai_rag_tool(question: str):
    """
    Use this tool to answer questions based on the context provided. Input should be a fully formed question.
    """
    
    response = graph.invoke({"question": question})
    
    # Return BOTH the answer AND the retrieved contexts
    # Format: JSON string with answer and contexts
    result = {
        "answer": response["response"],
        "contexts": [doc.page_content for doc in response["context"]]
    }
    
    return json.dumps(result)


In [ ]:
# Test the updated ai_rag_tool to verify it returns contexts
test_result = ai_rag_tool.invoke({"question": "Who is Amit Shrigondekar?"})
print("Tool result:", test_result[:200], "...")

# Parse it to see the structure
import json
parsed = json.loads(test_result)
print("\nParsed structure:")
print(f"- answer: {parsed['answer'][:100]}...")
print(f"- contexts: {len(parsed['contexts'])} chunks retrieved")
print(f"- First context: {parsed['contexts'][0][:100]}...")


In [17]:
# lets create a tool belt

tools = [ai_rag_tool, arxiv_tool, tavily_tool]

In [18]:
# Bind tools to the llm
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1",temperature=0)
model=model.bind_tools(tools)

model.invoke("who is Amit Shrigondekar?")

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5Wn0U99h9VYwovESsu9oVF2i', 'function': {'arguments': '{"question":"Who is Amit Shrigondekar?"}', 'name': 'ai_rag_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 197, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_422e2d36a8', 'id': 'chatcmpl-CRmlyTKpJWhvpVmHnVsjlq1eFXnHP', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--3e7a0d12-9ccb-4002-94d2-50bffc70a74f-0', tool_calls=[{'name': 'ai_rag_tool', 'args': {'question': 'Who is Amit Shrigondekar?'}, 'id': 'call_5Wn0U99h9VYwovESsu9oVF2i', 'type': 'tool_call'}], usage_metadata={'input_tokens': 197, 'output_tokens': 23, 'tot

In [19]:
# Langgraph Agent

from typing import TypedDict, Annotated,List
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages:Annotated[list,add_messages]
    context:List[Document]


In [20]:
from langgraph.prebuilt import ToolNode

def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

In [21]:
from langgraph.graph import StateGraph,END

uncompiled_graph= StateGraph(AgentState)
uncompiled_graph.add_node("agent",call_model)
uncompiled_graph.add_node("action",tool_node)




In [22]:
# add function/runnable for the conditional edge

def should_continue(state):
    last_message=state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return END 


uncompiled_graph.set_entry_point("agent")
uncompiled_graph.add_conditional_edges(
    "agent", should_continue
)
uncompiled_graph.add_edge("action","agent");
    

In [23]:
compiled_graph=uncompiled_graph.compile()


In [24]:
# Run the graph
from langchain_core.messages import HumanMessage

inputs={"messages":[HumanMessage(content="who is Amit Shrigondekar?")]}

async for chunk in compiled_graph.astream(inputs,stream_mode="updates"):
    for node,values in chunk.items():
        print("Receiving updates from node",node)
        print(values["messages"])
        print("\n\n")
    

Receiving updates from node agent
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_stVkNnbNqzVJpEbmzbn5DtUK', 'function': {'arguments': '{"question":"Who is Amit Shrigondekar?"}', 'name': 'ai_rag_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 197, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_e24a1fec47', 'id': 'chatcmpl-CRmm0yU7fsuVupGFQaX7ye4qChQLg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--eb6121aa-55c3-460d-b0ba-7823a36f3ab3-0', tool_calls=[{'name': 'ai_rag_tool', 'args': {'question': 'Who is Amit Shrigondekar?'}, 'id': 'call_stVkNnbNqzVJpEbmzbn5DtUK', 'type': 'tool_call'}], usage_metadata={'input_toke

# Ragas Baseline And SDG

In [25]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())



In [27]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/65 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Property 'summary' already exists in node 'fab977'. Skipping!
Property 'summary' already exists in node '612c9b'. Skipping!
Property 'summary' already exists in node '6e0974'. Skipping!
Property 'summary' already exists in node '159963'. Skipping!
Property 'summary' already exists in node 'eaa9c1'. Skipping!
Property 'summary' already exists in node 'ccc2fa'. Skipping!
Property 'summary' already exists in node 'b6f283'. Skipping!
Property 'summary' already exists in node '2f9748'. Skipping!
Property 'summary' already exists in node 'c5e9c6'. Skipping!
Property 'summary' already exists in node '703173'. Skipping!
Property 'summary' already exists in node '9194b6'. Skipping!
Property 'summary' already exists in node '8abd6a'. Skipping!
Property 'summary' already exists in node '68aaf9'. Skipping!
Property 'summary' already exists in node '5e82b7'. Skipping!
Property 'summary' already exists in node '3126b8'. Skipping!
Property 'summary' already exists in node 'eeacc9'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '612c9b'. Skipping!
Property 'summary_embedding' already exists in node 'eaa9c1'. Skipping!
Property 'summary_embedding' already exists in node '6e0974'. Skipping!
Property 'summary_embedding' already exists in node 'fab977'. Skipping!
Property 'summary_embedding' already exists in node 'ccc2fa'. Skipping!
Property 'summary_embedding' already exists in node 'b6f283'. Skipping!
Property 'summary_embedding' already exists in node '159963'. Skipping!
Property 'summary_embedding' already exists in node '9194b6'. Skipping!
Property 'summary_embedding' already exists in node 'c5e9c6'. Skipping!
Property 'summary_embedding' already exists in node '68aaf9'. Skipping!
Property 'summary_embedding' already exists in node '8abd6a'. Skipping!
Property 'summary_embedding' already exists in node '5e82b7'. Skipping!
Property 'summary_embedding' already exists in node '2f9748'. Skipping!
Property 'summary_embedding' already exists in node '703173'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [28]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How has Artificial Intelligence contributed to...,[Introduction ChatGPT launched in November 202...,ChatGPT is based on a Large Language Model (LL...,single_hop_specifc_query_synthesizer
1,how artificial intelligence like chatgpt grow ...,[Introduction ChatGPT launched in November 202...,"ChatGPT, based on a Large Language Model (LLM)...",single_hop_specifc_query_synthesizer
2,How is Generative AI like ChatGPT used in know...,[Conclusion This paper studies the rapid growt...,ChatGPT is used in knowledge-intensive jobs pr...,single_hop_specifc_query_synthesizer
3,What does Brynjolfsson estimate about the econ...,[Conclusion This paper studies the rapid growt...,Collis and Brynjolfsson (2025) estimate that U...,single_hop_specifc_query_synthesizer
4,How did the rapid global diffusion of new tech...,[<1-hop>\n\nConclusion This paper studies the ...,The rapid global diffusion of new technology s...,multi_hop_abstract_query_synthesizer
5,How do work-related versus non-work-related ch...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"According to the ChatGPT user data, as of July...",multi_hop_abstract_query_synthesizer
6,How do the ChatGPT adoption and usage statisti...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
7,What do the ChatGPT adoption and usage statist...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,ChatGPT experienced unprecedented rapid growth...,multi_hop_abstract_query_synthesizer
8,How did ChatGPT's user base and message volume...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had grown to more than 7...",multi_hop_specific_query_synthesizer
9,Considering the rapid adoption of ChatGPT by 7...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer


In [39]:
for test_row in dataset:
  inputs={"messages":[HumanMessage(content=test_row.eval_sample.user_input)]}
  response = compiled_graph.invoke(inputs)
  test_row.eval_sample.response = response["messages"][-1].content;
  # test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [40]:
dataset.to_pandas()

,user_input,reference_contexts,response,reference,synthesizer_name
0,How has Artificial Intelligence contributed to...,[Introduction ChatGPT launched in November 202...,Artificial Intelligence (AI) has played a cent...,ChatGPT is based on a Large Language Model (LL...,single_hop_specifc_query_synthesizer
1,how artificial intelligence like chatgpt grow ...,[Introduction ChatGPT launched in November 202...,Great question! Here’s a clear explanation:\n\...,"ChatGPT, based on a Large Language Model (LLM)...",single_hop_specifc_query_synthesizer
2,How is Generative AI like ChatGPT used in know...,[Conclusion This paper studies the rapid growt...,Generative AI models like ChatGPT are increasi...,ChatGPT is used in knowledge-intensive jobs pr...,single_hop_specifc_query_synthesizer
3,What does Brynjolfsson estimate about the econ...,[Conclusion This paper studies the rapid growt...,Erik Brynjolfsson estimates that the economic ...,Collis and Brynjolfsson (2025) estimate that U...,single_hop_specifc_query_synthesizer
4,How did the rapid global diffusion of new tech...,[<1-hop>\n\nConclusion This paper studies the ...,The rapid global diffusion of new technology—e...,The rapid global diffusion of new technology s...,multi_hop_abstract_query_synthesizer
5,How do work-related versus non-work-related ch...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,According to recent ChatGPT user data (notably...,"According to the ChatGPT user data, as of July...",multi_hop_abstract_query_synthesizer
6,How do the ChatGPT adoption and usage statisti...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,Here’s a synthesis of the latest data and rese...,ChatGPT launched in November 2022 and by July ...,multi_hop_abstract_query_synthesizer
7,What do the ChatGPT adoption and usage statist...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,Here’s what the latest statistics and studies ...,ChatGPT experienced unprecedented rapid growth...,multi_hop_abstract_query_synthesizer
8,How did ChatGPT's user base and message volume...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT's user base and message ...","By July 2025, ChatGPT had grown to more than 7...",multi_hop_specific_query_synthesizer
9,Considering the rapid adoption of ChatGPT by 7...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Recent studies, including a major working pape...","By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer


In [ ]:
import json
import re
from typing import Any, List, Dict, Optional

class ContextParser:
    """Parser to extract contexts from tool call results in agent messages."""
    
    @staticmethod
    def detect_tool_type(message: Any) -> Optional[str]:
        """Detect which tool was called based on the message."""
        try:
            # Check if message has tool_calls attribute
            if hasattr(message, 'tool_calls') and message.tool_calls:
                tool_name = message.tool_calls[0].get('name', '')
                if 'tavily' in tool_name.lower():
                    return 'tavily'
                elif 'arxiv' in tool_name.lower():
                    return 'arxiv'
                elif 'ai_rag' in tool_name.lower() or 'rag' in tool_name.lower():
                    return 'ai_rag'
            
            # Check if it's a ToolMessage with a name attribute
            if hasattr(message, 'name'):
                tool_name = message.name
                if 'tavily' in tool_name.lower():
                    return 'tavily'
                elif 'arxiv' in tool_name.lower():
                    return 'arxiv'
                elif 'ai_rag' in tool_name.lower() or 'rag' in tool_name.lower():
                    return 'ai_rag'
                    
        except Exception as e:
            print(f"Error detecting tool type: {e}")
        return None
    
    @staticmethod
    def extract_content(message: Any) -> Optional[str]:
        """Extract content from a message object."""
        try:
            if hasattr(message, 'content'):
                content = message.content
                if isinstance(content, str):
                    # Handle JSON-like strings with quotes
                    if content.startswith('content="') or content.startswith("content='"):
                        # Find the matching end quote by counting quotes
                        stack = []
                        in_quote = False
                        for i, char in enumerate(content):
                            if char == '"' and (i == 0 or content[i-1] != '\\'):
                                if not in_quote:
                                    stack.append(i)
                                    in_quote = True
                                else:
                                    stack.pop()
                                    if not stack and 'name=' in content[i:]:
                                        return content[8:i]  # 8 is len('content="')
                    return content
        except Exception as e:
            print(f"Error extracting content: {e}")
        return None
    
    @staticmethod
    def parse_tavily_results(content: str) -> List[Dict]:
        """Parse Tavily search results from content string."""
        try:
            # Parse the JSON string
            if isinstance(content, str):
                data = json.loads(content)
                
                # Get results array from the appropriate location
                if isinstance(data, list):
                    results = data
                elif isinstance(data, dict):
                    if 'artifact' in data and 'results' in data['artifact']:
                        results = data['artifact']['results']
                    elif 'results' in data:
                        results = data['results']
                    else:
                        results = []
                else:
                    results = []
                
                # Ensure content is string in each result
                for result in results:
                    if isinstance(result.get('content'), dict):
                        result['content'] = str(result['content'])
                        
                return results
        except Exception as e:
            print(f"Error parsing Tavily results: {str(e)}")
        return []
    
    @staticmethod
    def parse_ai_rag_results(content: str) -> List[Dict]:
        """Parse AI RAG tool results from content string."""
        try:
            # The ai_rag_tool now returns JSON with 'answer' and 'contexts'
            data = json.loads(content)
            
            # Extract contexts list
            if isinstance(data, dict) and 'contexts' in data:
                contexts = data['contexts']
                # Convert to list of dicts with 'content' key for consistency
                documents = [{'content': ctx} for ctx in contexts if ctx]
                return documents
            
            # Fallback: if it's just a plain string (old format)
            elif isinstance(content, str) and content.strip():
                return [{'content': content}]
            
            return []
        except json.JSONDecodeError:
            # If it's not JSON, treat it as plain text
            print("ai_rag content is not JSON, treating as plain text")
            return [{'content': content}] if content else []
        except Exception as e:
            print(f"Error parsing AI RAG results: {str(e)}")
            return []
    
    @staticmethod  
    def parse_arxiv_results(content: str) -> List[Dict]:
        """Parse arXiv results from content string."""
        try:
            papers = []
            # Split by "Published: " but keep the delimiter
            sections = re.split(r'(?=Published: )', content)
            
            for section in sections:
                if not section.strip():
                    continue
                
                # Extract paper details
                date_match = re.search(r'Published: (\d{4}-\d{2}-\d{2})', section)
                title_match = re.search(r'Title: (.*?)(?=\nAuthors:|$)', section, re.DOTALL)
                authors_match = re.search(r'Authors: (.*?)(?=\nSummary:|$)', section, re.DOTALL)
                summary_match = re.search(r'Summary: (.*?)(?=\n\nPublished:|$)', section, re.DOTALL)
                
                if date_match and title_match:
                    summary = summary_match.group(1).strip() if summary_match else ''
                    # Ensure summary is a string
                    if isinstance(summary, dict):
                        summary = str(summary)
                    
                    paper = {
                        'date': date_match.group(1),
                        'title': title_match.group(1).strip(),
                        'authors': authors_match.group(1).strip() if authors_match else '',
                        'summary': summary
                    }
                    papers.append(paper)
            
            return papers
        except Exception as e:
            print(f"Error parsing arXiv results: {str(e)}")
        return []


def parse_tool_call(message: Any) -> List[Dict]:
    """Parse tool call results from a conversation message."""
    parser = ContextParser()
    
    print("\n=== Starting message parsing ===")
    print(f"Message type: {type(message)}")
    
    # Detect tool type
    tool_type = parser.detect_tool_type(message)
    print(f"Detected tool type: {tool_type}")
    
    if tool_type is None:
        print("Could not detect tool type")
        return []
    
    # Extract content
    content = parser.extract_content(message)
    if content is None:
        print("Could not extract content")
        return []
    
    print(f"Extracted content type: {type(content)}")
    
    # Parse based on tool type
    if tool_type == 'tavily':
        results = parser.parse_tavily_results(content)
        print(f"Parsed {len(results)} Tavily results")
        return results
    elif tool_type == 'ai_rag':
        results = parser.parse_ai_rag_results(content)
        print(f"Parsed {len(results)} AI RAG results")
        return results
    elif tool_type == 'arxiv':
        results = parser.parse_arxiv_results(content)
        print(f"Parsed {len(results)} arXiv results")
        return results
    
    return []


In [44]:
def extract_contexts_from_messages(messages: List) -> List[str]:
    """
    Extract all contexts from a list of messages in a conversation.
    This looks for ToolMessage objects and parses their content.
    """
    all_contexts = []
    
    for message in messages:
        # Check if it's a ToolMessage (contains tool call results)
        if hasattr(message, 'name') and hasattr(message, 'content'):
            print(f"\nProcessing ToolMessage from {message.name}")
            
            # Parse the tool call results
            parsed_results = parse_tool_call(message)
            
            # Extract content from parsed results
            for result in parsed_results:
                if isinstance(result, dict):
                    # Different tools return different formats
                    if 'content' in result:
                        all_contexts.append(result['content'])
                    elif 'summary' in result:
                        all_contexts.append(result['summary'])
                    elif 'text' in result:
                        all_contexts.append(result['text'])
    
    return all_contexts


def get_contexts_from_graph_response(response: dict) -> List[str]:
    """
    Extract contexts from a graph response that contains messages.
    """
    if 'messages' not in response:
        print("No messages found in response")
        return []
    
    contexts = extract_contexts_from_messages(response['messages'])
    print(f"\nExtracted {len(contexts)} total contexts")
    return contexts


In [45]:
# Updated evaluation loop with context extraction
for test_row in dataset:
    inputs = {"messages": [HumanMessage(content=test_row.eval_sample.user_input)]}
    response = compiled_graph.invoke(inputs)
    
    # Extract the response text from the last message
    test_row.eval_sample.response = response["messages"][-1].content
    
    # Extract contexts from tool calls in the conversation
    retrieved_contexts = get_contexts_from_graph_response(response)
    
    # Store contexts for evaluation
    test_row.eval_sample.retrieved_contexts = retrieved_contexts
    
    print(f"Question: {test_row.eval_sample.user_input[:100]}...")
    print(f"Retrieved {len(retrieved_contexts)} contexts")
    print("-" * 50)



Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.human.HumanMessage'>
Error detecting tool type: 'NoneType' object has no attribute 'lower'
Detected tool type: None
Could not detect tool type

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.ai.AIMessage'>
Error detecting tool type: 'NoneType' object has no attribute 'lower'
Detected tool type: None
Could not detect tool type

Extracted 0 total contexts
Question: How has Artificial Intelligence contributed to the rapid adoption of ChatGPT?...
Retrieved 0 contexts
--------------------------------------------------

Processing ToolMessage from None

=== Starting message parsing ===
Message type: <class 'langchain_core.messages.human.HumanMessage'>
Error detecting tool type: 'NoneType' object has no attribute 'lower'
Detected tool type: None
Could not detect tool type

Processing ToolMessage from None

=== Start

In [46]:
# Verify dataset is properly populated
df = dataset.to_pandas()
print("Dataset columns:", df.columns.tolist())
print("\nSample row check:")
sample = df.iloc[0]
print(f"- user_input: {type(sample['user_input'])} - {str(sample['user_input'])[:100]}...")
print(f"- response: {type(sample['response'])} - {str(sample['response'])[:100]}...")
print(f"- reference: {type(sample['reference'])} - {str(sample['reference'])[:100]}...")
print(f"- retrieved_contexts: {type(sample['retrieved_contexts'])} - Length: {len(sample['retrieved_contexts']) if sample['retrieved_contexts'] else 0}")

# Check if all required columns have data
print("\nData completeness check:")
print(f"- Rows with user_input: {df['user_input'].notna().sum()}/{len(df)}")
print(f"- Rows with response: {df['response'].notna().sum()}/{len(df)}")
print(f"- Rows with reference: {df['reference'].notna().sum()}/{len(df)}")
print(f"- Rows with retrieved_contexts: {df['retrieved_contexts'].notna().sum()}/{len(df)}")


Dataset columns: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference', 'synthesizer_name']

Sample row check:
- user_input: <class 'str'> - How has Artificial Intelligence contributed to the rapid adoption of ChatGPT?...
- response: <class 'str'> - Artificial Intelligence (AI) has played a central role in the rapid adoption of ChatGPT. Here are th...
- reference: <class 'str'> - ChatGPT is based on a Large Language Model (LLM), a type of Artificial Intelligence developed over t...
- retrieved_contexts: <class 'list'> - Length: 0

Data completeness check:
- Rows with user_input: 12/12
- Rows with response: 12/12
- Rows with reference: 12/12
- Rows with retrieved_contexts: 12/12


In [47]:
# Run RAGAS evaluation with properly populated dataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper

# Create evaluation dataset from pandas
evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

# Configure evaluation
custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# Run evaluation with all metrics
baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),      # Requires: user_input, retrieved_contexts, reference
        Faithfulness(),          # Requires: user_input, response, retrieved_contexts
        FactualCorrectness(),    # Requires: user_input, response, reference
        ResponseRelevancy(),     # Requires: user_input, response
        ContextEntityRecall(),   # Requires: user_input, retrieved_contexts, reference
        NoiseSensitivity()       # Requires: user_input, retrieved_contexts
    ],
    llm=evaluator_llm,
    run_config=custom_run_config
)

# Display results
print("RAGAS Evaluation Results:")
print(baseline_result)


Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[5]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[11]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[17]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[59]: TimeoutError()


RAGAS Evaluation Results:
{'context_recall': 0.4015, 'faithfulness': 0.6503, 'factual_correctness': 0.4383, 'answer_relevancy': 0.9589, 'context_entity_recall': 0.1602, 'noise_sensitivity_relevant': 0.3581}


In [48]:
print(baseline_result)

{'context_recall': 0.4015, 'faithfulness': 0.6503, 'factual_correctness': 0.4383, 'answer_relevancy': 0.9589, 'context_entity_recall': 0.1602, 'noise_sensitivity_relevant': 0.3581}


## Let us use Cohere's Rerank model for re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!

In [72]:
# Let us build the adjusted retriever using cohere's rerank model

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance,VectorParams
from langchain_openai import OpenAIEmbeddings

text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100)
split_docs = text_splitter.split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="cohere_rag_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="cohere_rag_collection",
    embedding=embeddings,
)
vector_store.add_documents(documents=split_docs)
cohere_retriever = vector_store.as_retriever(search_kwargs={"k": 20})


from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def cohere_adjusted_retriever(state):
  compressor = CohereRerank(model="rerank-v3.5",top_n=5)
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=cohere_retriever
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}



# Lets build a graph 
from langgraph.graph import START,StateGraph

# 
cohere_graph_builder = StateGraph(State).add_sequence([cohere_adjusted_retriever, generate])
cohere_graph_builder.add_edge(START, "cohere_adjusted_retriever")
cohere_graph=cohere_graph_builder.compile()

# lets create a tool for the graph  
from langchain_core.tools import tool
import json

@tool
def ai_rag_tool_cohere(question: str):
    """
    Use this tool to answer questions based on the context provided. Input should be a fully formed question.
    """
    
    response = cohere_graph.invoke({"question": question})
    
    # Return BOTH the answer AND the retrieved contexts
    # Format: JSON string with answer and contexts
    result = {
        "answer": response["response"],
        "contexts": [doc.page_content for doc in response["context"]]
    }
    
    return json.dumps(result)

tools_cohere = [ai_rag_tool_cohere, arxiv_tool, tavily_tool]

# Bind tools to the llm
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1",temperature=0)
model=model.bind_tools(tools_cohere)


from langgraph.prebuilt import ToolNode

def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

tool_node_cohere = ToolNode(tools_cohere)

from langgraph.graph import StateGraph,END

cohere_uncompiled_graph= StateGraph(AgentState)
cohere_uncompiled_graph.add_node("agent",call_model)
cohere_uncompiled_graph.add_node("action",tool_node_cohere)

def should_continue(state):
    last_message=state["messages"][-1]
    if last_message.tool_calls:
        return "action"
    return END 


cohere_uncompiled_graph.set_entry_point("agent")
cohere_uncompiled_graph.add_conditional_edges(
    "agent", should_continue
)
cohere_uncompiled_graph.add_edge("action","agent");

cohere_compiled_graph=cohere_uncompiled_graph.compile()

# Run the graph
from langchain_core.messages import HumanMessage

inputs={"messages":[HumanMessage(content="who is Amit Shrigondekar?")]}

async for chunk in cohere_compiled_graph.astream(inputs,stream_mode="updates"):
    for node,values in chunk.items():
        print("Receiving updates from node",node)
        print(values["messages"])
        print("\n\n")




Receiving updates from node agent
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_m2ux53yAThfGE8KyZG1SKrFO', 'function': {'arguments': '{"query":"Amit Shrigondekar"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 198, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_564354cebb', 'id': 'chatcmpl-CSCV7Mt83JrkfPceehepFYM9uG18e', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--8b005d15-d3c5-48ee-9518-b6c82770c32d-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Amit Shrigondekar'}, 'id': 'call_m2ux53yAThfGE8KyZG1SKrFO', 'type': 'tool_call'}], usage_metadata={'in

In [ ]:
# import time
# import copy

# rerank_dataset = copy.deepcopy(dataset)

# # Updated evaluation loop with context extraction
# for test_row in rerank_dataset:
#     inputs = {"messages": [HumanMessage(content=test_row.eval_sample.user_input)]}
#     response = cohere_compiled_graph.invoke(inputs)
    
#     # Extract the response text from the last message
#     test_row.eval_sample.response = response["messages"][-1].content
    
#     # Extract contexts from tool calls in the conversation
#     retrieved_contexts = get_contexts_from_graph_response(response)
    
#     # Store contexts for evaluation
#     test_row.eval_sample.retrieved_contexts = retrieved_contexts
    
#     print(f"Question: {test_row.eval_sample.user_input[:100]}...")
#     print(f"Retrieved {len(retrieved_contexts)} contexts")
#     print("-" * 50)
#     time.sleep(2) # To try to avoid rate limiting.

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

# Updated evaluation loop with context extraction
for test_row in rerank_dataset:
    inputs = {"messages": [HumanMessage(content=test_row.eval_sample.user_input)]}
    response = cohere_compiled_graph.invoke(inputs)
    
    # DEBUG: Print which tools were called
    print(f"\n=== Question: {test_row.eval_sample.user_input[:80]}... ===")
    for msg in response["messages"]:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"Tool called: {tool_call.get('name', 'unknown')}")
        if hasattr(msg, 'name') and msg.name:
            print(f"Tool response from: {msg.name}")
    
    # Extract the response text from the last message
    test_row.eval_sample.response = response["messages"][-1].content
    
    # Extract contexts from tool calls in the conversation
    retrieved_contexts = get_contexts_from_graph_response(response)
    
    # Store contexts for evaluation
    test_row.eval_sample.retrieved_contexts = retrieved_contexts
    
    print(f"Retrieved {len(retrieved_contexts)} contexts")
    print("-" * 50)
    time.sleep(2)

In [74]:
# Test what ai_rag_tool_cohere actually returns
test_result = ai_rag_tool_cohere.invoke({"question": "What is ChatGPT?"})
print("=== RAW TOOL OUTPUT ===")
print(f"Type: {type(test_result)}")
print(f"Content: {test_result}")
print("\n=== TRYING TO PARSE AS JSON ===")
try:
    import json
    parsed = json.loads(test_result)
    print(f"Parsed successfully: {parsed}")
    print(f"Has 'contexts' key: {'contexts' in parsed}")
    if 'contexts' in parsed:
        print(f"Number of contexts: {len(parsed['contexts'])}")
        print(f"First context: {parsed['contexts'][0][:100] if parsed['contexts'] else 'EMPTY'}")
except Exception as e:
    print(f"ERROR: {e}")

=== RAW TOOL OUTPUT ===
Type: <class 'str'>
Content: {"answer": "ChatGPT is the first mass-market chatbot and likely the largest, based on a Large Language Model (LLM), a type of Artificial Intelligence (AI) developed over the last decade that represents an acceleration in AI capabilities. It allows users to submit plain-text messages (\"prompts\") and returns generated text-based responses. While additional features like web search and image generation exist, the most typical interaction remains the exchange of text messages. ChatGPT is used widely for producing writing, software code, spreadsheets, and other digital products, providing customized responses and novel content generation that distinguish it from traditional web search engines.", "contexts": ["1\nIntroduction\nChatGPT launched in November 2022. By July 2025, 18 billion messages were being sent each week\nby 700 million users, representing around 10% of the global adult population.1 For a new technology,\nthis speed of gl

In [76]:
# Verify dataset is properly populated
df = rerank_dataset.to_pandas()
print("Dataset columns:", df.columns.tolist())
print("\nSample row check:")
sample = df.iloc[0]
print(f"- user_input: {type(sample['user_input'])} - {str(sample['user_input'])[:100]}...")
print(f"- response: {type(sample['response'])} - {str(sample['response'])[:100]}...")
print(f"- reference: {type(sample['reference'])} - {str(sample['reference'])[:100]}...")
print(f"- retrieved_contexts: {type(sample['retrieved_contexts'])} - Length: {len(sample['retrieved_contexts']) if sample['retrieved_contexts'] else 0}")

# Check if all required columns have data
print("\nData completeness check:")
print(f"- Rows with user_input: {df['user_input'].notna().sum()}/{len(df)}")
print(f"- Rows with response: {df['response'].notna().sum()}/{len(df)}")
print(f"- Rows with reference: {df['reference'].notna().sum()}/{len(df)}")
print(f"- Rows with retrieved_contexts: {df['retrieved_contexts'].notna().sum()}/{len(df)}")

Dataset columns: ['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference', 'synthesizer_name']

Sample row check:
- user_input: <class 'str'> - How has Artificial Intelligence contributed to the rapid adoption of ChatGPT?...
- response: <class 'str'> - Artificial Intelligence (AI) has played a central role in the rapid adoption of ChatGPT, and its con...
- reference: <class 'str'> - ChatGPT is based on a Large Language Model (LLM), a type of Artificial Intelligence developed over t...
- retrieved_contexts: <class 'list'> - Length: 0

Data completeness check:
- Rows with user_input: 12/12
- Rows with response: 12/12
- Rows with reference: 12/12
- Rows with retrieved_contexts: 12/12


In [77]:
# Run RAGAS evaluation with properly populated dataset
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig
from ragas import EvaluationDataset
from ragas.llms import LangchainLLMWrapper

# Create evaluation dataset from pandas
cohere_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

# Configure evaluation
custom_run_config = RunConfig(timeout=360)
cohere_evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

# Run evaluation with all metrics
cohere_rerank_result = evaluate(
    dataset=cohere_evaluation_dataset,
    metrics=[
        LLMContextRecall(),      # Requires: user_input, retrieved_contexts, reference
        Faithfulness(),          # Requires: user_input, response, retrieved_contexts
        FactualCorrectness(),    # Requires: user_input, response, reference
        ResponseRelevancy(),     # Requires: user_input, response
        ContextEntityRecall(),   # Requires: user_input, retrieved_contexts, reference
        NoiseSensitivity()       # Requires: user_input, retrieved_contexts
    ],
    llm=cohere_evaluator_llm,
    run_config=custom_run_config
)

# Display results
print("RAGAS Evaluation Results:")
print(cohere_rerank_result)

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[23]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[35]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[5]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[11]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[17]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[59]: ValueError(zero-size array to reduction operation maximum which has no identity)
Exception raised in Job[65]: TimeoutError()


RAGAS Evaluation Results:
{'context_recall': 0.2574, 'faithfulness': 0.4566, 'factual_correctness': 0.5017, 'answer_relevancy': 0.9563, 'context_entity_recall': 0.0807, 'noise_sensitivity_relevant': 0.4254}


In [78]:
print(cohere_rerank_result)

{'context_recall': 0.2574, 'faithfulness': 0.4566, 'factual_correctness': 0.5017, 'answer_relevancy': 0.9563, 'context_entity_recall': 0.0807, 'noise_sensitivity_relevant': 0.4254}


In [59]:
print(baseline_result)

{'context_recall': 0.4015, 'faithfulness': 0.6503, 'factual_correctness': 0.4383, 'answer_relevancy': 0.9589, 'context_entity_recall': 0.1602, 'noise_sensitivity_relevant': 0.3581}
